In [ ]:
import os
import sys
from huggingface_hub import snapshot_download
import torch
import numpy as np

# Add the src folder to Python's module search path
sys.path.append(os.path.abspath('../src/Calibrated_RF'))
from Processor import POWDERRF_Processor
from Modeling import Conv1D_RF_Classifier, Conv1D_RF_MultiClassifier
from train import trainer, evaluator
from Calibration import FNR_Calibrator

device = "cuda" if torch.cuda.is_available() else "cpu"

### Load Data

In [ ]:
rounds = (1,2,3)
gains = (90, 80, 70, 60)
download_folders = [
    f"Round{r}_Gain{g}" for r in rounds for g in gains
]

allow_patterns = [f"{folder}/*" for folder in download_folders] if download_folders else None
data_root = snapshot_download(
    repo_id="T-Arshad/POWDER_CoChannel_Protocol_Dataset",
    repo_type="dataset",
    local_dir="POWDER_CoChannel_Protocol",
    allow_patterns=allow_patterns,
)

if download_folders:
    datadirs = [os.path.join(data_root, f).replace("\\", "/") for f in download_folders]
else:
    datadirs = [
        os.path.join(data_root, f"Round{r}_Gain{g}").replace("\\", "/")
        for r in (1, 2, 3) for g in (60, 70, 80, 90)
    ]

print("datadirs:")
for d in datadirs:
    print(" ", d, "exists:", os.path.isdir(d), "n_files:", len(os.listdir(d)) if os.path.isdir(d) else 0)

### Single Label RF-Fingerprinting

In [ ]:
Processor = POWDERRF_Processor(datadirs = datadirs, remove_tx = [], must_have_tx = [] , max_tx = 1, min_tx = 1)
Processor.file_samples = 20000000
xtrain, ytrain, xtest, ytest = Processor(n_samples = 20000, sample_len = 1024, signal_type = "fft")

In [ ]:
savedir = "./powder_fingerprint_model/"
if os.path.isdir(savedir) != True:
        os.mkdir(savedir)
model = Conv1D_RF_Classifier(classes = 6)
model.train()
model = trainer(model = model, xtrain = xtrain, ytrain = ytrain,  batch_sz = 256, lr = 0.001
              ,epochs = 25 , savepath = savedir, device = device)
model.eval()
eval_results = evaluator(model = model, xtest = xtest, ytest = ytest, savepath = savedir, device = device)

### Multi Label RF-Fingerprinting

In [ ]:
Processor = POWDERRF_Processor(datadirs = datadirs, remove_tx = [], must_have_tx = [] ,max_tx = float("inf"), min_tx = 1)
Processor.file_samples = 20000000
xtrain, ytrain, xtest, ytest = Processor(n_samples = 20000, sample_len = 1024, signal_type = "fft")

In [ ]:
savedir = "./powder_cochannelfingerprint_model/"
if os.path.isdir(savedir) != True:
        os.mkdir(savedir)
model = Conv1D_RF_MultiClassifier(classes = 6)
model.train()
model.conf_thresh = 0.50
model = trainer(model = model, xtrain = xtrain, ytrain = ytrain,  batch_sz = 256, lr = 0.001
              ,epochs = 25 , savepath = savedir, device = device)
model.eval()
eval_results = evaluator(model = model, xtest = xtest, ytest = ytest, savepath = savedir, device = device)

#### Calibration and Evaluation Over All Data

In [ ]:
Processor = POWDERRF_Processor(datadirs = datadirs, remove_tx = [], must_have_tx = [] ,max_tx = 6, min_tx = 1)
Processor.file_samples = 20000000
x_calib, y_calib, x_test, y_test = Processor(n_samples = 15000, sample_len = 1024, signal_type = "fft", train_test_split = 0.66666)
alpha = [0.05, 0.15, 0.25]
model.eval()
FNR_dict = FNR_Calibrator(model = model, x_calib = x_calib, y_calib = y_calib, alpha = alpha, lambda_step_sz = 0.01)
model.conf_thresh = 1 - FNR_dict["calib"][0][1]
eval_results_05 = evaluator(model = model, xtest = x_test, ytest = y_test, savepath = savedir, device = device)
model.conf_thresh = 1 - FNR_dict["calib"][1][1]
eval_results_15 = evaluator(model = model, xtest = x_test, ytest = y_test, savepath = savedir, device = device)
model.conf_thresh = 1 - FNR_dict["calib"][2][1]
eval_results_25 = evaluator(model = model, xtest = x_test, ytest = y_test, savepath = savedir, device = device)

#### Calibration and Evaluation at Gain = 90

In [ ]:
data_90 = [data for data in datadirs if "90" in data]
Processor = POWDERRF_Processor(datadirs = data_90, remove_tx = [], must_have_tx = [] ,max_tx = float("inf"), min_tx = 1)
Processor.file_samples = 20000000
x_calib, y_calib, x_test, y_test = Processor(n_samples = 15000, sample_len = 1024, signal_type = "fft", train_test_split = 0.66666, sample_index = [10e6, 20e6])
alpha = [0.05, 0.15, 0.25]
model.eval()
FNR_dict = FNR_Calibrator(model = model, x_calib = x_calib, y_calib = y_calib, alpha = alpha, lambda_step_sz = 0.01)
model.conf_thresh = 1 - FNR_dict["calib"][0][1]
eval_results_05 = evaluator(model = model, xtest = x_test, ytest = y_test, savepath = savedir, device = device)
model.conf_thresh = 1 - FNR_dict["calib"][1][1]
eval_results_15 = evaluator(model = model, xtest = x_test, ytest = y_test, savepath = savedir, device = device)
model.conf_thresh = 1 - FNR_dict["calib"][2][1]
eval_results_25 = evaluator(model = model, xtest = x_test, ytest = y_test, savepath = savedir, device = device)

#### Calibration and Evaluation at For 3 Occupants

In [ ]:
Processor = POWDERRF_Processor(datadirs = datadirs, remove_tx = [], must_have_tx = [] ,max_tx = 3, min_tx = 3)
Processor.file_samples = 20000000
x_calib, y_calib, x_test, y_test = Processor(n_samples = 15000, sample_len = 1024, signal_type = "fft", train_test_split = 0.66666)
alpha = [0.05, 0.15, 0.25]
model.eval()
FNR_dict = FNR_Calibrator(model = model, x_calib = x_calib, y_calib = y_calib, alpha = alpha, lambda_step_sz = 0.01)
model.conf_thresh = 1 - FNR_dict["calib"][0][1]
eval_results_05 = evaluator(model = model, xtest = x_test, ytest = y_test, savepath = savedir, device = "cuda")
model.conf_thresh = 1 - FNR_dict["calib"][1][1]
eval_results_15 = evaluator(model = model, xtest = x_test, ytest = y_test, savepath = savedir, device = "cuda")
model.conf_thresh = 1 - FNR_dict["calib"][2][1]
eval_results_25 = evaluator(model = model, xtest = x_test, ytest = y_test, savepath = savedir, device = "cuda")

##### Training, Calibration, and Evaluation with intruder

In [ ]:
intruder = 0

In [ ]:
Processor = POWDERRF_Processor(datadirs = datadirs, remove_tx = [intruder], must_have_tx = [] ,max_tx = float("inf"), min_tx = 1)
Processor.file_samples = 20000000
xtrain, ytrain, xtest, ytest = Processor(n_samples = 20000, sample_len = 1024, signal_type = "fft")

In [ ]:
savedir = "./powder_cochannelfingerprint_model/"
if os.path.isdir(savedir) != True:
        os.mkdir(savedir)
model = Conv1D_RF_MultiClassifier(classes = 5)
model.train()
model.conf_thresh = 0.50
model = trainer(model = model, xtrain = xtrain, ytrain = ytrain,  batch_sz = 256, lr = 0.001
              ,epochs = 25 , savepath = savedir, device = device)
model.eval()
eval_results = evaluator(model = model, xtest = xtest, ytest = ytest, savepath = savedir, device = device)

In [ ]:
Processor = POWDERRF_Processor(datadirs = datadirs, remove_tx = [intruder], must_have_tx = [] ,max_tx = 6, min_tx = 1)
Processor.file_samples = 20000000
x_calib, y_calib, x_test, y_test = Processor(n_samples = 5000, sample_len = 1024, signal_type = "fft", train_test_split = 1.0)

intruder_Processor = POWDERRF_Processor(datadirs = datadirs, remove_tx = [], must_have_tx = [intruder] ,max_tx = 6, min_tx = 1)
intruder_Processor.file_samples = 20000000
x_test, y_test, _, _ = intruder_Processor(n_samples = 10000, sample_len = 1024, signal_type = "fft", train_test_split = 1.0)
y_test = np.delete(y_test, intruder, axis = 1)
alpha = [0.05, 0.15, 0.25]
model.eval()
FNR_dict = FNR_Calibrator(model = model, x_calib = x_calib, y_calib = y_calib, alpha = alpha, lambda_step_sz = 0.01)
model.conf_thresh = 1 - FNR_dict["calib"][0][1]
eval_results_05 = evaluator(model = model, xtest = x_test, ytest = y_test, savepath = savedir, device = "cuda")
model.conf_thresh = 1 - FNR_dict["calib"][1][1]
eval_results_15 = evaluator(model = model, xtest = x_test, ytest = y_test, savepath = savedir, device = "cuda")
model.conf_thresh = 1 - FNR_dict["calib"][2][1]
eval_results_25 = evaluator(model = model, xtest = x_test, ytest = y_test, savepath = savedir, device = "cuda")